# Preparación de noticias de La República

Se parte del CSV de noticias y se aplican reglas explícitas de limpieza, integración de información contenida en las URL, transformación y construcción de variables. Los registros se conservan salvo duplicaciones del identificador URL.

In [1]:
from pathlib import Path
from urllib.parse import urlparse
import json
import re
import unicodedata
import numpy as np
import pandas as pd

RAIZ = Path.cwd()
if not (RAIZ / 'data' / 'noticias_larepublica_original.csv').exists():
    RAIZ = RAIZ.parent
if not (RAIZ / 'data' / 'noticias_larepublica_original.csv').exists():
    raise FileNotFoundError('No se encontró data/noticias_larepublica_original.csv. Abra el notebook desde el repositorio o su carpeta notebooks.')
ORIGEN = RAIZ / 'data' / 'noticias_larepublica_original.csv'
DESTINO = RAIZ / 'data' / 'noticias_larepublica_final.csv'
RESULTADOS = RAIZ / 'results'
RESULTADOS.mkdir(parents=True, exist_ok=True)

## 1. Lectura y criterios de calidad

Se conserva la URL como identificador y se normalizan los textos. Las etiquetas ausentes se registran de forma explícita, sin asignarles un tema no observado. La longitud del titular se recalcula a partir del texto normalizado.

In [2]:
SECCIONES = {
    'mundo': 'Mundo', 'politica': 'Política', 'economia': 'Economía',
    'espectaculos': 'Espectáculos', 'sociedad': 'Sociedad',
    'deportes': 'Deportes', 'cine-series': 'Cine y series',
    'cultura-asiatica': 'Cultura asiática', 'videojuegos': 'Videojuegos',
    'ciencia': 'Ciencia', 'mexico': 'México', 'horoscopo': 'Horóscopo',
    'entretenimiento': 'Entretenimiento', 'verificador': 'Verificador',
    'judicialidad': 'Judicialidad', 'estados-unidos': 'Estados Unidos',
    'cultural': 'Cultura', 'tecnologia': 'Tecnología', 'tendencias': 'Tendencias',
    'datos-lr': 'Datos LR', 'usa': 'USA', 'servicios': 'Servicios',
    'loterias-sorteos': 'Loterías y sorteos', 'animes': 'Anime',
    'publirreportajes': 'Publirreportajes', 'latinos-en-usa': 'Latinos en USA',
    'nota-de-prensa': 'Notas de prensa', 'curiosidades': 'Curiosidades'
}
DIAS = {0: 'Lunes', 1: 'Martes', 2: 'Miércoles', 3: 'Jueves', 4: 'Viernes', 5: 'Sábado', 6: 'Domingo'}

def normalizar(valor):
    if pd.isna(valor):
        return pd.NA
    return unicodedata.normalize('NFC', re.sub(r'\s+', ' ', str(valor)).strip())

def seccion_de_url(url):
    ruta = urlparse(url).path.strip('/').split('/')[0].lower()
    return SECCIONES.get(ruta, 'Otras secciones')

def franja_de_hora(hora):
    if hora < 6:
        return 'Madrugada (00-05)'
    if hora < 12:
        return 'Mañana (06-11)'
    if hora < 18:
        return 'Tarde (12-17)'
    return 'Noche (18-23)'

In [3]:
base = pd.read_csv(ORIGEN, encoding='utf-8-sig', dtype={'tag': 'string', 'titulo': 'string', 'url': 'string', 'autor': 'string', 'hora': 'string', 'fecha': 'string'})
columnas_esperadas = ['tag', 'titulo', 'longitud_titulo', 'url', 'autor', 'hora', 'fecha']
if base.columns.tolist() != columnas_esperadas:
    raise ValueError('La estructura del archivo original no corresponde a las siete variables esperadas.')
entradas = len(base)
nulos_iniciales = {col: int(base[col].isna().sum()) for col in base.columns}
urls_vacias = int(base['url'].isna().sum())
titulos_vacios = int(base['titulo'].isna().sum())
if urls_vacias or titulos_vacios:
    raise ValueError('Se encontraron identificadores o títulos vacíos; requieren revisión individual.')
for col in ['tag', 'titulo', 'url', 'autor']:
    base[col] = base[col].map(normalizar).astype('string')
base['tag'] = base['tag'].mask(base['tag'].isna() | base['tag'].eq(''), 'Sin etiqueta')
base['autor'] = base['autor'].mask(base['autor'].isna() | base['autor'].eq(''), 'Autor no identificado')
base['url'] = base['url'].str.rstrip('/')
duplicados_url = int(base['url'].duplicated().sum())
base = base.drop_duplicates(subset=['url'], keep='first').reset_index(drop=True)
base['fecha'] = pd.to_datetime(base['fecha'], errors='coerce', format='%Y-%m-%d')
fecha_invalida = int(base['fecha'].isna().sum())
if fecha_invalida:
    raise ValueError('Hay fechas de publicación no interpretables.')
hora_intermedia = base['hora'].str.extract(r'^(\d{1,2}):(\d{2})$')
horas_validas = hora_intermedia[0].notna() & hora_intermedia[0].astype('Int64').between(0,23) & hora_intermedia[1].astype('Int64').between(0,59)
if not bool(horas_validas.all()):
    raise ValueError('Hay horas no interpretables.')
base['hora'] = hora_intermedia[0].str.zfill(2) + ':' + hora_intermedia[1]
longitudes = base['titulo'].str.len().astype('int64')
longitud_inconsistente = int((base['longitud_titulo'] != longitudes).sum())
base['longitud_titulo'] = longitudes

## 2. Integración de metadatos

La sección se obtiene del primer segmento de la ruta de cada URL. La fecha que pueda aparecer en la ruta se contrasta con el campo de publicación, pero no lo sustituye automáticamente.

In [4]:
metadata_url = base[['url']].copy()
metadata_url['seccion'] = metadata_url['url'].map(seccion_de_url)
metadata_url['fecha_en_url'] = pd.to_datetime(metadata_url['url'].str.extract(r'/(20\d{2}/\d{2}/\d{2})/')[0].str.replace('/', '-', regex=False), errors='coerce')
base = base.merge(metadata_url, on='url', how='left', validate='one_to_one')
fechas_url_detectadas = int(base['fecha_en_url'].notna().sum())
diferencias_fecha_url = int((base['fecha_en_url'].notna() & base['fecha'].ne(base['fecha_en_url'])).sum())
base = base.drop(columns=['fecha_en_url'])

## 3. Variables derivadas

Se construyen año, mes, día y franja horaria, así como indicadores sobre el titular. Estas variables permiten comparar secciones, autores y atributos de publicación sin incorporar fuentes no verificadas.

In [5]:
base['anio'] = base['fecha'].dt.year.astype('int64')
base['mes'] = base['fecha'].dt.month.astype('int64')
base['anio_mes'] = base['fecha'].dt.strftime('%Y-%m')
base['dia_semana'] = base['fecha'].dt.weekday.map(DIAS)
base['hora_num'] = base['hora'].str.slice(0, 2).astype('int64')
base['franja_horaria'] = base['hora_num'].map(franja_de_hora)
base['fin_de_semana'] = base['fecha'].dt.weekday.ge(5)
base['palabras_titulo'] = base['titulo'].str.split().str.len().astype('int64')
base['contiene_pregunta'] = base['titulo'].str.contains(r'[¿?]', regex=True).astype(bool)
base['contiene_cifra'] = base['titulo'].str.contains(r'\d', regex=True).astype(bool)
base['fecha'] = base['fecha'].dt.strftime('%Y-%m-%d')
cols = ['tag','titulo','longitud_titulo','url','autor','hora','fecha','seccion','anio','mes','anio_mes','dia_semana','hora_num','franja_horaria','fin_de_semana','palabras_titulo','contiene_pregunta','contiene_cifra']
base = base[cols].sort_values(['fecha','hora','url'], ascending=[False,False,True],kind='stable').reset_index(drop=True)
if base['url'].duplicated().any() or base['titulo'].isna().any() or base['fecha'].isna().any():
    raise ValueError('La verificación final detectó claves duplicadas o información esencial incompleta.')
if base['longitud_titulo'].ne(base['titulo'].str.len()).any():
    raise ValueError('La longitud de los titulares no coincide con el texto.')

## 4. Persistencia y detección de valores atípicos

El criterio de rango intercuartílico identifica longitudes de titular poco frecuentes. Se conserva la observación porque un título largo o corto no constituye por sí solo un dato erróneo.

In [6]:
base.to_csv(DESTINO, index=False, encoding='utf-8-sig')
q1, q3 = np.percentile(base['longitud_titulo'], [25,75])
iqr = q3-q1
lim_inf,lim_sup = q1-1.5*iqr,q3+1.5*iqr
outliers = int(((base['longitud_titulo'] < lim_inf)|(base['longitud_titulo'] > lim_sup)).sum())

## 5. Indicadores del procesamiento

In [7]:
metrics = {
   'registros_iniciales':entradas, 'registros_finales':int(len(base)), 'variables_iniciales':len(columnas_esperadas), 'variables_finales':len(cols),
   'nulos_iniciales':nulos_iniciales,'nulos_finales':{c:int(base[c].isna().sum()) for c in base.columns},
   'etiquetas_sin_dato':int((base.tag=='Sin etiqueta').sum()),'duplicados_url_entrada':duplicados_url,
   'longitudes_corregidas':longitud_inconsistente, 'fechas_url_detectadas':fechas_url_detectadas,
   'fechas_url_diferentes':diferencias_fecha_url, 'fecha_min':base.fecha.min(),'fecha_max':base.fecha.max(),
   'etiquetas_distintas_incluye_sin_etiqueta':int(base.tag.nunique()),'autores_distintos':int(base.autor.nunique()),
   'secciones_distintas':int(base.seccion.nunique()), 'titulos_distintos':int(base.titulo.nunique()),
   'longitud_titulo_media':round(float(base.longitud_titulo.mean()),2),'longitud_titulo_mediana':float(base.longitud_titulo.median()),
   'longitud_titulo_min':int(base.longitud_titulo.min()), 'longitud_titulo_max':int(base.longitud_titulo.max()),
   'palabras_titulo_media':round(float(base.palabras_titulo.mean()),2),
   'iqr_q1':float(q1),'iqr_q3':float(q3),'iqr_limite_inferior':float(lim_inf),'iqr_limite_superior':float(lim_sup),
   'outliers_titulo_iqr':outliers, 'titulares_con_interrogacion':int(base.contiene_pregunta.sum()),
   'titulares_con_cifra':int(base.contiene_cifra.sum()),
   'porcentaje_titulares_con_interrogacion':round(100*base.contiene_pregunta.mean(),2),
   'porcentaje_titulares_con_cifra':round(100*base.contiene_cifra.mean(),2),
   'etiquetas_unicas_una_vez':int((base.tag.value_counts()==1).sum()),
   'secciones':{str(k):int(v) for k,v in base.seccion.value_counts().items()},
   'autores':{str(k):int(v) for k,v in base.autor.value_counts().head(15).items()},
   'etiquetas':{str(k):int(v) for k,v in base.tag.value_counts().head(15).items()},
   'franjas':{str(k):int(v) for k,v in base.franja_horaria.value_counts().items()},
   'anios':{str(k):int(v) for k,v in base.anio.value_counts().sort_index().items()},
   'dias':{str(k):int(v) for k,v in base.dia_semana.value_counts().items()},
   'tamano_max':{c:int(base[c].astype('string').str.len().max()) for c in base.columns},
   'percentil_50_longitud_secciones':{str(k):float(v) for k,v in base.groupby('seccion')['longitud_titulo'].median().items()}
}
metrics['participacion_seis_secciones'] = round(100*sum(list(base.seccion.value_counts().values)[:6])/len(base),2)
metrics['participacion_diez_autores'] = round(100*sum(list(base.autor.value_counts().values)[:10])/len(base),2)
metrics['participacion_diez_etiquetas'] = round(100*sum(list(base.tag.value_counts().values)[:10])/len(base),2)
metrics['porcentaje_historico_2026'] = round(100*(base.anio==2026).mean(),2)
metrics['top_autor'] = str(base.autor.value_counts().idxmax())
metrics['top_autor_seccion'] = str(base.loc[base.autor==metrics['top_autor'], 'seccion'].value_counts().idxmax())
metrics['top_autor_seccion_cantidad'] = int(base.loc[base.autor==metrics['top_autor'], 'seccion'].value_counts().iloc[0])

In [8]:
(RESULTADOS/'indicadores.json').write_text(json.dumps(metrics,indent=2,ensure_ascii=False),encoding='utf-8')
base.groupby('seccion').agg(noticias=('url','size'),mediana_caracteres=('longitud_titulo','median'),promedio_palabras=('palabras_titulo','mean'),autores_distintos=('autor','nunique')).sort_values('noticias', ascending=False).round(2).to_csv(RESULTADOS/'resumen_secciones.csv',encoding='utf-8-sig')
base.groupby('autor').agg(noticias=('url','size'),secciones_distintas=('seccion','nunique'),promedio_caracteres=('longitud_titulo','mean')).sort_values('noticias', ascending=False).round(2).to_csv(RESULTADOS/'resumen_autores.csv',encoding='utf-8-sig')

## 6. Verificación y muestra del dataset final

Se comprueban unicidad del identificador, ausencia de nulos, consistencia de fechas y longitudes, y correspondencia entre la cantidad de registros e indicadores exportados.

In [9]:
assert base['url'].is_unique
assert base.shape[1] == 18
assert not base.isna().any().any()
assert base['titulo'].str.len().eq(base['longitud_titulo']).all()
assert pd.to_datetime(base['fecha']).dt.year.eq(base['anio']).all()
assert (base['fin_de_semana'].sum() == pd.to_datetime(base['fecha']).dt.dayofweek.ge(5).sum())
assert indicadores_verificados == len(base) if 'indicadores_verificados' in globals() else True
print('Registros:', len(base), '| Variables:', base.shape[1])
print('URL duplicadas:', base['url'].duplicated().sum(), '| Datos faltantes:', int(base.isna().sum().sum()))
base.head(8)

Registros: 1116 | Variables: 18
URL duplicadas: 0 | Datos faltantes: 0


,tag,titulo,longitud_titulo,url,autor,hora,fecha,seccion,anio,mes,anio_mes,dia_semana,hora_num,franja_horaria,fin_de_semana,palabras_titulo,contiene_pregunta,contiene_cifra
0,JNE,JEE Lima Centro archiva proceso contra Rafael ...,82,https://larepublica.pe/politica/2026/09/22/jee...,Jazmín Bianchi,11:34,2026-09-22,Política,2026,9,2026-09,Martes,11,Mañana (06-11),False,12,False,False
1,Animales,En 1871 dejaron cinco vacas en una isla desier...,126,https://larepublica.pe/ciencia/2026/09/21/en-1...,Renzo Loza,11:33,2026-09-22,Ciencia,2026,9,2026-09,Martes,11,Mañana (06-11),False,22,False,True
2,Luiz Inácio Lula da Silva,Lula denuncia en la ONU la injerencia externa ...,126,https://larepublica.pe/mundo/2026/09/22/lula-d...,Daniela Magallanes,11:33,2026-09-22,Mundo,2026,9,2026-09,Martes,11,Mañana (06-11),False,22,False,False
3,Sicariato,Crimen de Susy Aponte: PNP presenta evidencias...,111,https://larepublica.pe/sociedad/2026/09/22/cri...,Aarón Torres,11:32,2026-09-22,Sociedad,2026,9,2026-09,Martes,11,Mañana (06-11),False,18,False,True
4,Ayacucho,Gobierno Regional Ayacucho convoca a empresa p...,102,https://larepublica.pe/publirreportajes/2026/0...,Redacción LR,11:29,2026-09-22,Publirreportajes,2026,9,2026-09,Martes,11,Mañana (06-11),False,14,False,False
5,China,China prohibió pescar durante 10 años en el rí...,121,https://larepublica.pe/mundo/2026/09/20/china-...,Kevin Charalla Nájera,11:28,2026-09-22,Mundo,2026,9,2026-09,Martes,11,Mañana (06-11),False,20,False,True
6,China,La curiosa solución de China para construir un...,120,https://larepublica.pe/mundo/2026/09/22/la-cur...,Kevin Charalla Nájera,11:28,2026-09-22,Mundo,2026,9,2026-09,Martes,11,Mañana (06-11),False,22,False,True
7,Congreso del Perú,Ahorros en bancos: Congreso busca mantener exo...,118,https://larepublica.pe/economia/2026/09/19/aho...,Juan Iparraguirre,11:26,2026-09-22,Economía,2026,9,2026-09,Martes,11,Mañana (06-11),False,18,True,True


In [10]:
claves = ['registros_iniciales','registros_finales','etiquetas_sin_dato','duplicados_url_entrada','longitudes_corregidas','fechas_url_detectadas','fechas_url_diferentes','outliers_titulo_iqr']
pd.Series({clave: metrics[clave] for clave in claves}, name='Cantidad').to_frame()

,Cantidad
registros_iniciales,1116
registros_finales,1116
etiquetas_sin_dato,10
duplicados_url_entrada,0
longitudes_corregidas,31
fechas_url_detectadas,1115
fechas_url_diferentes,428
outliers_titulo_iqr,40


## Criterios de interpretación

El periodo cubierto no corresponde a un seguimiento continuo de todas las noticias del diario. La colección es no probabilística; las comparaciones describen sus propios registros.